In [3]:
from models.encoder import Encoder
from models.cgan import Generator, Discriminator,disc_loss, gen_loss, weights_init
import torch
import torch.nn as nn
from torch.utils.data import DataLoader
from torchvision import datasets, transforms
import torch.optim as optim
import os
transform = transforms.Compose([
    transforms.Resize((64,64)),
    transforms.ToTensor(),
    transforms.Normalize((0.5,0.5,0.5),(0.5,0.5,0.5))
])
n_critics = 5
# dataset = datasets.ImageFolder(
#     root="/kaggle/input/datasets/olafkrastovski/handwritten-digits-0-9",
#     transform=transform
# )
# loader = DataLoader(dataset, batch_size=32, shuffle=True,num_workers=2,pin_memory=True,persistent_workers=True)
gen = Generator().to("cuda")
disc = Discriminator(10).to("cuda")
encoder = Encoder().to("cuda")
optimizer_gen  = optim.Adam(gen.parameters(),lr = 2e-4, betas=(0.0,0.9))
optimizer_disc  = optim.Adam(disc.parameters(),lr = 2e-4, betas=(0.0,0.9))
print(gen)
print(disc)


Generator(
  (input): ConvTranspose2d(100, 1024, kernel_size=(4, 4), stride=(1, 1))
  (cbn0): ConditionalBatchNorm2d(
    (bn): BatchNorm2d(1024, eps=1e-05, momentum=0.1, affine=False, track_running_stats=True)
    (embed): Embedding(10, 2048)
  )
  (block1): ResBlockUp(
    (cbn1): ConditionalBatchNorm2d(
      (bn): BatchNorm2d(1024, eps=1e-05, momentum=0.1, affine=False, track_running_stats=True)
      (embed): Embedding(10, 2048)
    )
    (cbn2): ConditionalBatchNorm2d(
      (bn): BatchNorm2d(512, eps=1e-05, momentum=0.1, affine=False, track_running_stats=True)
      (embed): Embedding(10, 1024)
    )
    (conv1): Conv2d(1024, 512, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (conv2): Conv2d(512, 512, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (upsample): Upsample(scale_factor=2.0, mode='nearest')
    (conv_sc): Conv2d(1024, 512, kernel_size=(1, 1), stride=(1, 1))
  )
  (block2): ResBlockUp(
    (cbn1): ConditionalBatchNorm2d(
      (bn): BatchNorm2d(512, ep

In [4]:
print(f"{torch.cuda.is_available()}")
print(f"Torch version: {torch.__version__}")

True
Torch version: 2.5.1


In [5]:
epochs = 50
discriminator_loss = []
generator_loss = []
for epoch in range(epochs):
    batch_loss_gen = []
    batch_loss_disc = []
    for batch_idx,(image,labels) in enumerate(loader):
        image = image.to("cuda")
        labels = labels.to("cuda")
        for _ in range(n_critics):
            optimizer_disc.zero_grad()
            loss_d = disc_loss(image.size(0),100,image,labels)
            loss_d.backward()
            optimizer_disc.step()
        optimizer_gen.zero_grad()
        loss_g = gen_loss(image.size(0),100,labels)
        loss_g.backward()
        optimizer_gen.step()
        if batch_idx % 100 == 0:

            print(
                f"Epoch [{epoch+1}/{epochs}] "
                f"Batch [{batch_idx}/{len(loader)}] "
                f"D Loss: {loss_d.item():.4f} "
                f"G Loss: {loss_g.item():.4f}"
            )
        batch_loss_gen.append(loss_g.item())
        batch_loss_disc.append(loss_d.item())
    discriminator_loss.append(sum(batch_loss_disc)/len(batch_loss_disc))
    generator_loss.append(sum(batch_loss_gen)/len(batch_loss_gen))
    with torch.no_grad():
        labels = torch.randint(9,(16,))
        print(labels)
        z = torch.randn(16,100,1,1).to("cuda")
        labels = labels.to("cuda")
        fake_images = gen(z,labels).detach().cpu()

        fake_images = (fake_images + 1) / 2

        grid = make_grid(fake_images, nrow=4)

        plt.figure(figsize=(8,8))
        plt.imshow(grid.permute(1,2,0))
        plt.axis("off")
        plt.title(f"Epoch {epoch+1}")
        plt.show()

NameError: name 'loader' is not defined